# 🎙️ Hima TTS Nano (62M) - Telugu & Hindi All-in-One Colab

This notebook lets you run **Hima TTS Nano (62M)** by Prahalad Reddy directly in Google Colab.

### Features included:
1. **Telugu Text-to-Speech** with 6 preset voices (*Bhavana, Lakshmi, Sravani, Srinivas, Karthik, Pranay*).
2. **Hindi Text-to-Speech** with 6 preset voices (*Meera, Sneha, Nandini, Siddharth, Ishaan, Nikhil*).
3. **Zero-Shot Voice Cloning** (Upload 5-10 seconds of any voice).
4. **Interactive Web UI (Gradio)** with a shareable public link.

*(Tip: Go to **Runtime > Change runtime type** and ensure **T4 GPU** is selected.)*

## ⚙️ Step 1: Install Dependencies & Download Models

In [ ]:
#@title 1. Setup Environment & Models (Click Play ▶️)
# Install base libraries
!pip install -q -U huggingface_hub speechbrain soundfile gradio ipywidgets

# Download Telugu & Hindi models directly via HuggingFace API
from huggingface_hub import snapshot_download

print("📥 Downloading Telugu model (hima-tts-nano-te)...")
snapshot_download(repo_id="Prahaladha/hima-tts-nano-te", local_dir="/content/hima-tts-nano-te")

print("📥 Downloading Hindi model (hima-tts-nano-hi)...")
snapshot_download(repo_id="Prahaladha/hima-tts-nano-hi", local_dir="/content/hima-tts-nano-hi")

# Install model dependencies
!pip install -q -r /content/hima-tts-nano-te/requirements.txt
!pip install -q -r /content/hima-tts-nano-hi/requirements.txt

print("\n Setup complete! Both models are downloaded and ready to use.")

## 🗣️ Step 2: Generate Telugu Speech

In [ ]:
#@title 2. Telugu TTS Generator
from IPython.display import Audio, display

telugu_text = "ఏమండీ, సాయంత్రం free గా ఉన్నారా? బయటికి వెళ్ళి coffee తాగుదాం." #@param {type:"string"}
telugu_voice = "Bhavana" #@param ["Bhavana", "Lakshmi", "Sravani", "Srinivas", "Karthik", "Pranay", "Random"]

voice_flag = f"--voice {telugu_voice}" if telugu_voice != "Random" else ""

%cd /content/hima-tts-nano-te
!python -m hima_tts "{telugu_text}" {voice_flag} -o /content/telugu_output.wav
%cd /content

display(Audio("/content/telugu_output.wav", autoplay=True))

## 🗣️ Step 3: Generate Hindi Speech

In [ ]:
#@title 3. Hindi TTS Generator
from IPython.display import Audio, display

hindi_text = "नमस्ते! आज का मौसम बहुत अच्छा है, आप कैसे हैं?" #@param {type:"string"}
hindi_voice = "Meera" #@param ["Meera", "Sneha", "Nandini", "Siddharth", "Ishaan", "Nikhil", "Random"]

voice_flag = f"--voice {hindi_voice}" if hindi_voice != "Random" else ""

%cd /content/hima-tts-nano-hi
!python -m hima_tts "{hindi_text}" {voice_flag} -o /content/hindi_output.wav
%cd /content

display(Audio("/content/hindi_output.wav", autoplay=True))

## 🧬 Step 4: Zero-Shot Voice Cloning (Your Own Voice)
Upload a 5 to 10 second clear `.wav` sample of any voice to clone it immediately!

In [ ]:
#@title 4. Upload & Clone Voice
from google.colab import files
from IPython.display import Audio, display
import os

print("Please upload a clean 5-10 second speech audio file (.wav format):")
uploaded = files.upload()

if uploaded:
    sample_file = list(uploaded.keys())[0]
    full_sample_path = os.path.abspath(sample_file)
    
    language = "Telugu" #@param ["Telugu", "Hindi"]
    clone_text = "ఇది నా సొంత వాయిస్ తో క్లోన్ చేయబడిన ఆడియో." #@param {type:"string"}
    
    target_dir = "/content/hima-tts-nano-te" if language == "Telugu" else "/content/hima-tts-nano-hi"
    output_file = "/content/cloned_output.wav"
    
    %cd {target_dir}
    !python -m hima_tts "{clone_text}" --voice "{full_sample_path}" -o "{output_file}"
    %cd /content
    
    print("Cloning complete! Listen below:")
    display(Audio(output_file, autoplay=True))
else:
    print("No file uploaded.")

## 🌐 Step 5: Launch Interactive Gradio Web App
Run the cell below to launch a clean Web UI with a public URL.

In [ ]:
#@title 5. Launch Web UI (Gradio)
import gradio as gr
import subprocess
import os

def generate_speech(language, text, voice, custom_audio):
    output_path = "/content/gradio_output.wav"
    if os.path.exists(output_path):
        os.remove(output_path)
        
    model_dir = "/content/hima-tts-nano-te" if language == "Telugu" else "/content/hima-tts-nano-hi"
    
    if custom_audio is not None:
        voice_arg = f'--voice "{custom_audio}"'
    elif voice and voice != "Random":
        voice_arg = f'--voice {voice}'
    else:
        voice_arg = ""
        
    cmd = f'cd {model_dir} && python -m hima_tts "{text}" {voice_arg} -o "{output_path}"'
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    if os.path.exists(output_path):
        return output_path, res.stdout
    else:
        return None, f"Error: {res.stderr}"

with gr.Blocks(title="Hima TTS Nano Web UI") as demo:
    gr.Markdown("# 🎙️ Hima TTS Nano (62M) Multi-Voice & Cloning")
    
    with gr.Row():
        with gr.Column():
            lang = gr.Radio(["Telugu", "Hindi"], label="Language", value="Telugu")
            txt = gr.Textbox(label="Text to Speak", value="ఏమండీ, సాయంత్రం free గా ఉన్నారా?", lines=3)
            voice = gr.Dropdown(["Bhavana", "Lakshmi", "Sravani", "Srinivas", "Karthik", "Pranay", "Meera", "Sneha", "Nandini", "Siddharth", "Ishaan", "Nikhil", "Random"], label="Preset Voice", value="Bhavana")
            audio_ref = gr.Audio(label="(Optional) Zero-Shot Voice Sample (5-10s .wav)", type="filepath")
            btn = gr.Button("Generate Audio", variant="primary")
        with gr.Column():
            out_audio = gr.Audio(label="Generated Speech")
            logs = gr.Textbox(label="Logs / Output", lines=5)
            
    btn.click(generate_speech, inputs=[lang, txt, voice, audio_ref], outputs=[out_audio, logs])

demo.launch(share=True, debug=False)